## 04 — Embedding-based Models (Semantic Representations)

Goal: Evaluate embedding-based semantic representations for hallucination detection on HaluEval,
using the same data splits and evaluation protocol as prior baselines (Notebooks 02–03).


In [1]:
import sys
from pathlib import Path

# Ensure repo root is in PYTHONPATH when running from notebooks/
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [2]:
import numpy as np
import pandas as pd

from src.data.load_splits import load_splits
from src.features.embeddings import EmbeddingConfig, embed_dataframe
from src.models.embedding_baseline import build_embedding_lr
from src.utils import evaluate_split, metrics_table


In [3]:
train_df, val_df, test_df = load_splits(ROOT)

print(train_df.shape, val_df.shape, test_df.shape)
print(train_df.columns.tolist())


(51647, 7) (6425, 7) (6435, 7)
['id', 'group_id', 'task', 'prompt', 'response', 'label', 'context']


#### Quick sanity checks

In [4]:
print("Label balance (train):")
print(train_df["label"].value_counts(normalize=True))

print("\nResponse length (chars) summary (train):")
print(train_df["response"].astype(str).fillna("").str.len().describe())


Label balance (train):
label
0    0.534494
1    0.465506
Name: proportion, dtype: float64

Response length (chars) summary (train):
count    51647.000000
mean       190.356478
std        197.645575
min          1.000000
25%         44.000000
50%        106.000000
75%        290.000000
max       3381.000000
Name: response, dtype: float64


### MiniLM baseline
Embedding extraction (MiniLM) + cache + MPS

In [5]:
cfg = EmbeddingConfig(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    text_col="response",
    device="mps",                 # Mac acceleration (fallback to "cpu" if needed)
    batch_size=64,
    normalize=True,
    cache_dir="artifacts/embeddings",
)

X_train = embed_dataframe(train_df, cfg, cache_tag="train")
X_val   = embed_dataframe(val_df,   cfg, cache_tag="val")
X_test  = embed_dataframe(test_df,  cfg, cache_tag="test")

y_train = train_df["label"].values
y_val   = val_df["label"].values
y_test  = test_df["label"].values

print(X_train.shape, X_val.shape, X_test.shape)
print("nan?", np.isnan(X_train).any(), "inf?", np.isinf(X_train).any())


/Users/aviv.gross/hallu-detect/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 820.48it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1247.89it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+

(51647, 384) (6425, 384) (6435, 384)
nan? False inf? False


#### Train + Evaluate (MiniLM + LR)

In [6]:
model = build_embedding_lr()
model.fit(X_train, y_train)

m_train = evaluate_split("train_minilm", model, X_train, y_train)
m_val   = evaluate_split("val_minilm",   model, X_val,   y_val)
m_test  = evaluate_split("test_minilm",  model, X_test,  y_test)

m_train, m_val, m_test

/Users/aviv.gross/hallu-detect/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



train_minilm metrics:
  accuracy = 0.7656
  f1       = 0.7603
  precision= 0.7254
  recall   = 0.7986
  confusion matrix:
[[20338  7267]
 [ 4841 19201]]

val_minilm metrics:
  accuracy = 0.7647
  f1       = 0.7562
  precision= 0.7265
  recall   = 0.7885
  confusion matrix:
[[2568  883]
 [ 629 2345]]

test_minilm metrics:
  accuracy = 0.7598
  f1       = 0.7528
  precision= 0.7199
  recall   = 0.7889
  confusion matrix:
[[2535  916]
 [ 630 2354]]


(SplitMetrics(split='train_minilm', accuracy=0.7655623753557805, f1=0.760285092060978, precision=0.7254420432220039, recall=0.7986440395973713),
 SplitMetrics(split='val_minilm', accuracy=0.7646692607003891, f1=0.7562076749435666, precision=0.726456009913259, recall=0.7885003362474782),
 SplitMetrics(split='test_minilm', accuracy=0.7597513597513598, f1=0.7527982091461465, precision=0.7198776758409786, recall=0.7888739946380697))

In [7]:
metrics_table([m_train, m_val, m_test])

,split,accuracy,f1,precision,recall
0,train_minilm,0.765562,0.760285,0.725442,0.798644
1,val_minilm,0.764669,0.756208,0.726456,0.788500
2,test_minilm,0.759751,0.752798,0.719878,0.788874


### Error Analysis (MiniLM + LR)

In [8]:
def get_pred_proba(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        return (s - s.min()) / (s.max() - s.min() + 1e-9)
    return model.predict(X)

proba_test = get_pred_proba(model, X_test)
pred_test = (proba_test >= 0.5).astype(int)

err_df = test_df.copy()
err_df["pred"] = pred_test
err_df["proba"] = proba_test

fp = err_df[(err_df["label"] == 0) & (err_df["pred"] == 1)].sort_values("proba", ascending=False)
fn = err_df[(err_df["label"] == 1) & (err_df["pred"] == 0)].sort_values("proba", ascending=True)

print("FP:", fp.shape, "FN:", fn.shape)


FP: (916, 9) FN: (630, 9)


#### Display FP / FN examples

In [9]:
def preview_text(s, n=300):
    s = "" if s is None else str(s)
    return s[:n] + ("..." if len(s) > n else "")

def make_error_table(df, n=10):
    out = df.head(n).copy()
    out["prompt_snip"] = out["prompt"].apply(preview_text)
    out["response_snip"] = out["response"].apply(preview_text)
    cols = ["task", "label", "pred", "proba", "prompt_snip", "response_snip"]
    return out[cols]

print("Top False Positives (confident but wrong):")
display(make_error_table(fp, n=10))

print("Top False Negatives (missed hallucinations):")
display(make_error_table(fn, n=10))


Top False Positives (confident but wrong):


,task,label,pred,proba,prompt_snip,response_snip
1764,dialogue,0,1,0.983933,[Human]: Sia Furler is someone I am not at all...,"Well, Prince comes to mind. He could play ever..."
108,dialogue,0,1,0.978800,[Human]: Venus Williams is probably one of the...,"I had no idea, what is that about?"
1614,dialogue,0,1,0.968023,[Human]: I like The Lost World: Jurassic Park ...,"The Goonies was written, edited and produced b..."
1830,dialogue,0,1,0.964851,[Human]: Could you recommend an author related...,John Steelye was another author o The Adventur...
28,dialogue,0,1,0.962115,[Human]: I like actor Mike Starr. Could you re...,It's both. It appears to be almost making fun...
44,dialogue,0,1,0.956288,[Human]: Could you recommend any movies direct...,"Perhaps you would like Larry Crowne, written b..."
4102,summarization,0,1,0.954278,When Bruce Jenner told ABC's Diane Sawyer and ...,Social media largely supports Jenner. More peo...
592,dialogue,0,1,0.954111,[Human]: I like watching basketball. Do you li...,Oh really? I didn't know that. That's a very i...
4438,summarization,0,1,0.954005,"Dikembe Mutombo, an eight-times NBA All-Star w...",Dikembe Mutombo was an eight-times NBA All-Sta...
618,dialogue,0,1,0.952476,[Human]: Do you know the Director Fernando Mei...,He's also worked on the TV series Antonia and ...


Top False Negatives (missed hallucinations):


,task,label,pred,proba,prompt_snip,response_snip
3383,qa,1,0,0.000748,"Narasimhavarman II, popularly known as Rajasim...",700 AD
2423,qa,1,0,0.006996,Tom and Jerry: A Nutcracker Tale is an example...,Cartoon
3373,qa,1,0,0.013993,What is the nationality of the person who pres...,Italian
2527,qa,1,0,0.017380,The 2010 FIFA World Cup Final had a single goa...,Real Madrid
953,dialogue,1,0,0.018038,[Human]: Do you have movies with the actor Tom...,Windsor McKay
3513,qa,1,0,0.024970,What kind of royalty does Hussa bint Ahmed Al ...,Nobility
5333,summarization,1,0,0.026815,Ethan Czahor (pictured) has launched ‘Clear’ w...,New app 'Clear' developed by Ethan Czahor help...
2577,qa,1,0,0.027553,Surinder Sodhi composed the music for special ...,"January 5, 2013"
2959,qa,1,0,0.027895,"Which singer, born in 1977, shared the stage w...",Rhonda Lea Vincent
2041,dialogue,1,0,0.031488,[Human]: What could you recommend by L.M. Mont...,In 1928


## Error Analysis — MiniLM + Logistic Regression

### False Positives (label = 0, pred = 1)
The model frequently flags non-hallucinatory responses as hallucinations in the following cases:

- **Conversational or non-informative responses**:  
  Short, casual answers (e.g., acknowledgments, refusals, or jokes) that contain little factual content are often classified as hallucinations, despite not making any factual claims.

- **Vague or generic factual statements**:  
  Responses that sound encyclopedic or general but lack concrete details or sources tend to be over-flagged. Semantically weak or abstract phrasing appears similar to hallucination patterns in the embedding space.

- **Task-dependent ambiguity**:  
  False positives are especially common in *dialogue* and *summarization* tasks, where factual grounding is less explicit and “truth” is harder to define, leading the model to confuse stylistic vagueness with factual incorrectness.

---

### False Negatives (label = 1, pred = 0)
The model fails to detect hallucinations primarily in the following scenarios:

- **Short, fluent factual answers**:  
  One-word or very short answers (e.g., names, dates, titles) that are factually incorrect but semantically plausible are often misclassified as non-hallucinatory.

- **Semantically coherent but incorrect responses**:  
  Answers that are linguistically fluent and contextually relevant, yet contain subtle factual errors, are difficult for the model to distinguish from correct responses based on embeddings alone.

- **Question-answering (QA) tasks**:  
  The majority of false negatives occur in QA settings, where hallucinations often manifest as confident but wrong factual assertions rather than stylistic or semantic anomalies.

---

### Interpretation
- **What embeddings capture well**:  
  Semantic coherence, topical relevance, and stylistic similarity to known hallucination patterns, leading to high recall in detecting suspicious or low-content responses.

- **What embeddings miss**:  
  Factual correctness and fine-grained verification, particularly for concise and confident answers that lack explicit uncertainty cues.

- **Hypothesis for improvement**:  
  Combining semantic embeddings with surface-level and uncertainty-aware features (e.g., response length, confidence markers, hedging language, and source presence) can help balance recall and precision by capturing complementary signals that embeddings alone fail to model.


## Hybrid (MiniLM + Numeric)

#### Add numeric features

In [10]:
from src.features import add_numeric_feature_columns, get_numeric_feature_cols
from src.models.embedding_baseline import concat_embeddings_and_numeric

train_df_num = add_numeric_feature_columns(train_df.copy())
val_df_num   = add_numeric_feature_columns(val_df.copy())
test_df_num  = add_numeric_feature_columns(test_df.copy())

num_cols = get_numeric_feature_cols(train_df_num)  # 👈 כאן התיקון
print("Number of numeric features:", len(num_cols))
print("First 10 numeric cols:", num_cols[:10])


Number of numeric features: 10
First 10 numeric cols: ['resp_n_chars', 'resp_n_words', 'resp_n_punct', 'resp_has_multi_excl', 'resp_has_multi_q', 'resp_has_ellipsis', 'resp_n_numbers', 'resp_n_uncertainty', 'resp_punct_per_word', 'resp_numbers_per_word']


#### Build mixed matrices

In [11]:
X_train_num = train_df_num[num_cols].values
X_val_num   = val_df_num[num_cols].values
X_test_num  = test_df_num[num_cols].values

X_train_mix = concat_embeddings_and_numeric(X_train, X_train_num)
X_val_mix   = concat_embeddings_and_numeric(X_val,   X_val_num)
X_test_mix  = concat_embeddings_and_numeric(X_test,  X_test_num)

print(X_train_mix.shape, X_val_mix.shape, X_test_mix.shape)
print("nan?", np.isnan(X_train_mix).any(), "inf?", np.isinf(X_train_mix).any())


(51647, 394) (6425, 394) (6435, 394)
nan? False inf? False


#### Train + Evaluate Hybrid

In [12]:
model_mix = build_embedding_lr()
model_mix.fit(X_train_mix, y_train)

mix_train = evaluate_split("train_minilm+num", model_mix, X_train_mix, y_train)
mix_val   = evaluate_split("val_minilm+num",   model_mix, X_val_mix,   y_val)
mix_test  = evaluate_split("test_minilm+num",  model_mix, X_test_mix,  y_test)

metrics_table([mix_train, mix_val, mix_test])


/Users/aviv.gross/hallu-detect/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



train_minilm+num metrics:
  accuracy = 0.7852
  f1       = 0.7795
  precision= 0.7463
  recall   = 0.8158
  confusion matrix:
[[20938  6667]
 [ 4429 19613]]

val_minilm+num metrics:
  accuracy = 0.7840
  f1       = 0.7745
  precision= 0.7492
  recall   = 0.8016
  confusion matrix:
[[2653  798]
 [ 590 2384]]

test_minilm+num metrics:
  accuracy = 0.7790
  f1       = 0.7712
  precision= 0.7418
  recall   = 0.8029
  confusion matrix:
[[2617  834]
 [ 588 2396]]


,split,accuracy,f1,precision,recall
0,train_minilm+num,0.785157,0.779500,0.746309,0.815781
1,val_minilm+num,0.783969,0.774529,0.749214,0.801614
2,test_minilm+num,0.779021,0.771162,0.741796,0.802949


In [13]:
rows = [
    {"model": "Embeddings(MiniLM)+LR",  "test_acc": m_test.accuracy,  "test_f1": m_test.f1,  "test_precision": m_test.precision,  "test_recall": m_test.recall},
    {"model": "Embeddings(MiniLM)+num", "test_acc": mix_test.accuracy,"test_f1": mix_test.f1,"test_precision": mix_test.precision,"test_recall": mix_test.recall},
]
df_cmp = pd.DataFrame(rows)
df_cmp


,model,test_acc,test_f1,test_precision,test_recall
0,Embeddings(MiniLM)+LR,0.759751,0.752798,0.719878,0.788874
1,Embeddings(MiniLM)+num,0.779021,0.771162,0.741796,0.802949


### Key findings
- MiniLM embeddings alone achieved ~0.77 test accuracy and ~0.76 F1, with relatively high recall but lower precision.
- Adding numeric surface/uncertainty features improved performance to ~0.79 test accuracy and ~0.78 F1, improving both precision and recall.
- Despite the improvement, the hybrid model still did not surpass the strongest classical baseline (~0.82), suggesting that HaluEval contains strong lexical/surface signals and that semantic representations alone are insufficient.

### Implication
Semantic representations provide complementary signal, but most predictive power appears to come from surface-level or lexical cues, motivating either richer lexical modeling or task-adapted fine-tuning as a next step.
